# COMP9444 26T2 — Project 005: Intelligent Food Image Recognition
## Exploratory Data Analysis — UECFoodPix

**Dataset:** [UECFoodPix](https://mm.cs.uec.ac.jp/uecfoodpix/)  ·  mask: bounding box + GrabCut  ·  semantic segmentation

---

| | |
|---|---|
| **Images** | 10,000 (9,000 train / 1,000 test) |
| **Classes** | 102 food + background = 103 output channels |
| **Images per class** | min 55, max 1,990, median 83 |
| **Task type** | 88% single-food per image → single-food segmentation |
| **Food coverage** | avg 42% food, 58% background per image |
| **Image size** | 80×71 ~ 800×800 px, median 420×338 |
| **Mask quality** | Auto-generated (GrabCut) — edges approximate, fine details may be missing |

## 1. Setup & Imports

In [ ]:
import os
import glob
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# ====== Plot settings ======
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (14, 7),
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.labelsize': 12,
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})

# Colour palette
PALETTE = sns.color_palette("viridis", as_cmap=False)
sns.set_palette(PALETTE)

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✓ Libraries imported")

## 2. Dataset Paths & Load Helpers

> 把 `DATA_ROOT` 改成你解压后数据集的实际路径

In [ ]:
# ====== CONFIGURE THIS PATH ======
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "UECFOODPIX" / "data"

CATEGORY_FILE = DATA_ROOT / "category.txt"
TRAIN_LIST = DATA_ROOT / "train9000.txt"
TEST_LIST = DATA_ROOT / "test1000.txt"

IMAGE_MASK_ROOT = DATA_ROOT / "UECFoodPIX"

TRAIN_IMG_DIR = IMAGE_MASK_ROOT / "train" / "img"
TRAIN_MASK_DIR = IMAGE_MASK_ROOT / "train" / "mask"
TEST_IMG_DIR = IMAGE_MASK_ROOT / "test" / "img"
TEST_MASK_DIR = IMAGE_MASK_ROOT / "test" / "mask"

print("Category exists:", CATEGORY_FILE.exists())
print("Train list exists:", TRAIN_LIST.exists())
print("Test list exists:", TEST_LIST.exists())
print("Train img exists:", TRAIN_IMG_DIR.exists())
print("Train mask exists:", TRAIN_MASK_DIR.exists())

In [ ]:
# ====== Parse category.txt ======
def load_categories(cat_file):
    """Load class ID → class name mapping."""
    id2name = {}
    with open(cat_file, 'r') as f:
        for line in f:
            if line.strip() == '' or line.startswith('id'):
                continue
            parts = line.strip().split(maxsplit=1)
            if len(parts) >= 2:
                class_id = int(parts[0])
                class_name = parts[1].strip()
                id2name[class_id] = class_name
    return id2name

ID2NAME = load_categories(CATEGORY_FILE)
NUM_CLASSES = len(ID2NAME)   # 103 (0 = background, 1–102 = food)

print(f"Total classes: {NUM_CLASSES}")
print(f"Class 0: {ID2NAME.get(0, 'N/A')}")
print(f"Class 1: {ID2NAME.get(1, 'N/A')}")
print(f"Class 102: {ID2NAME.get(102, 'N/A')}")
print(f"...\nFirst 10 classes:")
for cid in range(0, 10):
    print(f"  {cid:>3d} → {ID2NAME.get(cid, '?')}")

In [ ]:
# ====== Helper: read mask (R channel only) ======
def read_mask(mask_path):
    """Read segmentation mask. Only the R channel contains class IDs."""
    mask = np.array(Image.open(mask_path))
    if mask.ndim == 3:
        return mask[:, :, 0]           # R channel
    else:
        return mask                     # already grayscale


# ====== Helper: get all image + mask pairs ======
def get_image_mask_pairs(img_dir, mask_dir):
    """Return list of (img_path, mask_path) pairs."""
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    pairs = []
    for ip in img_paths:
        fname = os.path.splitext(os.path.basename(ip))[0]
        mp = os.path.join(mask_dir, fname + ".png")
        if os.path.exists(mp):
            pairs.append((ip, mp))
    return pairs

train_pairs = get_image_mask_pairs(TRAIN_IMG_DIR, TRAIN_MASK_DIR)
test_pairs  = get_image_mask_pairs(TEST_IMG_DIR, TEST_MASK_DIR)

print(f"Train pairs: {len(train_pairs)}")
print(f"Test  pairs: {len(test_pairs)}")
print(f"Total pairs: {len(train_pairs) + len(test_pairs)}")

---
## 3. Dataset Statistics

### 📊 Dataset Overview

In [ ]:
# ====== Build summary ======
num_train = len(train_pairs)
num_test  = len(test_pairs)
total     = num_train + num_test

summary_df = pd.DataFrame({
    "Split":   ["Train", "Test", "Total"],
    "Images":  [num_train, num_test, total],
    "Pct (%)": [
        f"{num_train/total*100:.1f}%",
        f"{num_test/total*100:.1f}%",
        "100%"
    ],
})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Pie chart ---
wedges, texts, autotexts = axes[0].pie(
    [num_train, num_test],
    labels=["Train", "Test"],
    autopct='%1.1f%%',
    colors=["#4C72B0", "#DD8452"],
    startangle=90,
    explode=(0.02, 0.02),
    textprops={'fontsize': 13}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title("Train / Test Split", fontweight='bold', fontsize=16)

# --- Bar chart ---
bars = axes[1].bar(
    ["Train", "Test"],
    [num_train, num_test],
    color=["#4C72B0", "#DD8452"],
    width=0.45,
    edgecolor='white',
    linewidth=1.5
)
for b, v in zip(bars, [num_train, num_test]):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 60, str(v),
                 ha='center', fontsize=14, fontweight='bold')
axes[1].set_title("Image Count per Split", fontweight='bold', fontsize=16)
axes[1].set_ylabel("# Images")
axes[1].set_ylim(0, num_train * 1.12)

plt.tight_layout()
plt.savefig("eda_dataset_overview.png", dpi=150)
plt.show()

# --- Print table ---
display(summary_df)

In [ ]:
# ====== Pre-compute class statistics (used throughout the notebook) ======
print("Scanning training masks for class statistics ...")

class_image_counter = Counter()   # how many images contain each class
class_pixel_counter = Counter()   # total pixels per class across all images
total_pixels_all = 0

for _, mask_path in tqdm(train_pairs, desc="Processing masks"):
    mask = read_mask(mask_path)
    ids, counts = np.unique(mask, return_counts=True)
    for cid, cnt in zip(ids, counts):
        class_pixel_counter[int(cid)] += int(cnt)
        class_image_counter[int(cid)] += 1
    total_pixels_all += mask.size

print(f"\u2713 Scanned {len(train_pairs)} masks, {total_pixels_all:,} total pixels")
print(f"  Unique class IDs found: {len(class_image_counter)} ({len(class_image_counter) - (1 if 0 in class_image_counter else 0)} food + background)")


---
## 4. Category Taxonomy

UECFoodPix has **102 food classes**. To help understand the task scope, I've grouped them into semantic categories (rice dishes, noodles, bread, meat, seafood, etc.).

This grouping helps us:
- Understand what types of food the model needs to distinguish
- Spot potential inter-class confusion (e.g., "rice" vs "chicken rice" vs "fried rice")
- Identify if certain food groups are under-represented


In [ ]:
# ====== Manual category grouping (based on Japanese food taxonomy) ======
CATEGORY_GROUPS = {
    "rice dishes":      list(range(1, 13)),    # 1–12
    "noodles":          list(range(13, 19)),   # 13–18
    "bread":            list(range(19, 25)),   # 19–24
    "meat dishes":      list(range(25, 38)),   # 25–37
    "seafood":          list(range(38, 46)),   # 38–45
    "egg / tofu":       list(range(46, 52)),   # 46–51
    "vegetables":       list(range(52, 62)),   # 52–61
    "soup / hotpot":    list(range(62, 67)),   # 62–66
    "sweets / dessert": list(range(67, 76)),   # 67–75
    "fruit":            list(range(76, 84)),   # 76–83
    "drinks":           list(range(84, 91)),   # 84–90
    "fast food":        list(range(91, 97)),   # 91–96
    "other":            list(range(97, 103)),  # 97–102
}

# Count images per group
group_img_counts = {}
print(f"{'Group':<18s} {'#Classes':>8s} {'#Images':>8s}")
print("-" * 38)
for gname, ids in CATEGORY_GROUPS.items():
    count = sum(class_image_counter.get(cid, 0) for cid in ids)
    group_img_counts[gname] = count
    print(f"  {gname:<16s} {len(ids):>8d} {count:>8d}")

# ====== Visualise ======
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

g_names = list(group_img_counts.keys())
g_vals  = list(group_img_counts.values())
g_palette = sns.color_palette("tab20", len(g_names))

bars = ax1.barh(range(len(g_names))[::-1], g_vals[::-1],
                color=g_palette[::-1], edgecolor="white", height=0.7)
ax1.set_yticks(range(len(g_names))[::-1])
ax1.set_yticklabels(g_names[::-1], fontsize=10)
ax1.set_xlabel("# Training Images (class appears in)", fontsize=12)
ax1.set_title("Images per Food Category Group", fontweight="bold", fontsize=14)
for b, v in zip(bars, g_vals[::-1]):
    ax1.text(b.get_width() + 10, b.get_y() + b.get_height()/2,
             str(v), va="center", fontsize=10, fontweight="bold")

# Examples per group
example_lines = []
for gname, ids in CATEGORY_GROUPS.items():
    foods = [ID2NAME.get(cid, '?') for cid in ids[:3]]
    example_lines.append(f"  {gname:<18s}  {', '.join(foods)}")

group_text = "CATEGORY GROUPS (sample classes)\n" + "\n".join(example_lines)
ax2.text(0.02, 0.98, group_text, transform=ax2.transAxes,
         fontsize=8.5, fontfamily="monospace", verticalalignment="top")
ax2.axis("off")
ax2.set_title("Sample Classes per Group", fontweight="bold", fontsize=14)

plt.tight_layout()
plt.savefig("eda_category_groups.png", dpi=150)
plt.show()

---
## 5. Image-Level Class Distribution

> 88% of images contain only ONE food class. Image-level counts — not pixel counts — are the correct metric for class balance in this dataset.

### 5.1 Images per Class

In [ ]:
# ====== Image-level class distribution (NOT pixel-level) ======
food_ids_only = sorted(ID2NAME.keys())  # 1–102
food_img_counts = [class_image_counter.get(cid, 0) for cid in food_ids_only]

# Sort by frequency
sorted_idx = np.argsort(food_img_counts)[::-1]
sorted_ids   = [food_ids_only[i] for i in sorted_idx]
sorted_names = [ID2NAME[cid] for cid in sorted_ids]
sorted_cnts  = [food_img_counts[i] for i in sorted_idx]

fig, axes = plt.subplots(2, 1, figsize=(22, 12))

# --- Top 30 ---
N_top = 30
top_colors = sns.color_palette("rocket_r", N_top)
axes[0].barh(range(N_top)[::-1], sorted_cnts[:N_top][::-1],
             color=top_colors, edgecolor="white", height=0.75)
axes[0].set_yticks(range(N_top)[::-1])
axes[0].set_yticklabels([f"{sorted_ids[i]} — {sorted_names[i]}" for i in range(N_top)][::-1], fontsize=9)
for i, v in enumerate(sorted_cnts[:N_top][::-1]):
    axes[0].text(v + 5, i, str(v), va="center", fontsize=9, fontweight="bold")
axes[0].set_xlabel("# Training Images containing this class", fontsize=12)
axes[0].set_title(f"Top {N_top} Classes by Image Count", fontweight="bold", fontsize=16)
axes[0].set_xlim(0, max(sorted_cnts[:N_top]) * 1.18)
axes[0].invert_yaxis()

# --- All 102 classes ---
mean_cnt = np.mean(food_img_counts)
median_cnt = np.median(food_img_counts)
axes[1].bar(food_ids_only, food_img_counts, color="#55A868", width=0.8, edgecolor="white", linewidth=0.3)
axes[1].axhline(y=mean_cnt, color="red", linestyle="--", linewidth=1.5,
                label=f"Mean: {mean_cnt:.0f} images/class")
axes[1].axhline(y=median_cnt, color="orange", linestyle="--", linewidth=1.5,
                label=f"Median: {median_cnt:.0f} images/class")
axes[1].set_xlabel("Class ID", fontsize=12)
axes[1].set_ylabel("# Training Images", fontsize=12)
axes[1].set_title("All 102 Food Classes — Image Count Distribution", fontweight="bold", fontsize=16)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150)
plt.show()


### 5.2 Class Balance Analysis (Image-Level)


In [ ]:
# ====== Image-level class balance analysis ======
tail_60 = sum(1 for c in food_img_counts if c < 60)
tail_80 = sum(1 for c in food_img_counts if c < 80)
ratio = max(food_img_counts) / max(min(food_img_counts), 1)

print(f"Images per class: max={max(food_img_counts)}  min={min(food_img_counts)}  "
      f"mean={np.mean(food_img_counts):.0f}  median={np.median(food_img_counts):.0f}")
print(f"Max/min ratio: {ratio:.0f}:1  |  < 60 images: {tail_60}/102  |  < 80 images: {tail_80}/102")

if ratio > 30:
    print(f"→ Moderate imbalance ({ratio:.0f}:1). Consider class-weighted loss for tail classes.")
elif ratio > 10:
    print(f"→ Mild imbalance ({ratio:.0f}:1). Standard CE may suffice; monitor tail-class IoU.")
else:
    print(f"→ Reasonably balanced ({ratio:.0f}:1). Standard loss is fine.")

In [ ]:
# ====== Image-level imbalance visualization ======
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

colors_img = ["#E74C3C" if c < 60 else "#F39C12" if c < 90 else "#55A868"
              for c in sorted_cnts]

ax1.bar(range(len(sorted_cnts)), sorted_cnts, color=colors_img, width=0.8)
ax1.set_yscale("log")
ax1.set_xlabel("Food Class (sorted by image count)", fontsize=12)
ax1.set_ylabel("# Training Images (log scale)", fontsize=12)
ax1.set_title("Image-Level Class Distribution (Log Scale)", fontweight="bold", fontsize=15)

legend_elements = [
    mpatches.Patch(color="#55A868", label="\u2265 90 images"),
    mpatches.Patch(color="#F39C12", label="60–89 images"),
    mpatches.Patch(color="#E74C3C", label="< 60 images (tail)"),
]
ax1.legend(handles=legend_elements, loc="upper right", fontsize=10)

# --- Balance pie ---
normal = len(food_img_counts) - tail_60
pie_data = [normal, tail_60]
pie_labels = [f"\u2265 60 images ({normal})", f"< 60 images ({tail_60})"]
pie_colors = ["#55A868", "#E74C3C"]

wedges, texts, autotexts = ax2.pie(
    pie_data, labels=pie_labels, autopct="%1.1f%%",
    colors=pie_colors, startangle=90, explode=(0.02, 0.06),
    textprops={"fontsize": 12})
for at in autotexts:
    at.set_fontweight("bold")
ax2.set_title("Tail Classes (< 60 images)", fontweight="bold", fontsize=15)

plt.tight_layout()
plt.savefig("eda_class_imbalance.png", dpi=150)
plt.show()


---
## 6. Per-Image Analysis

### 6.1 How Many Food Classes per Image?

This is critical: is this a **single-food** or **multi-food** segmentation dataset?


In [ ]:
# ====== Count food classes per image (full training set) ======
multi_food_counter = Counter()
for _, mask_path in tqdm(train_pairs, desc="Counting food classes per image"):
    mask = read_mask(mask_path)
    n_food = len([u for u in np.unique(mask) if u != 0])
    multi_food_counter[n_food] += 1

total_imgs = sum(multi_food_counter.values())
single_pct = multi_food_counter.get(1, 0) / total_imgs * 100
multi_pct = sum(v for k, v in multi_food_counter.items() if k >= 2) / total_imgs * 100

print(f"Food classes per image: 1→{single_pct:.0f}%  |  0→{multi_food_counter.get(0,0)/total_imgs*100:.1f}%  |  2+→{multi_pct:.1f}%")
print(f"→ {single_pct:.0f}% single-food. This is a single-food segmentation dataset.")

# ====== Visualise ======
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

labels = [str(k) for k in sorted(multi_food_counter.keys())]
values = [multi_food_counter[k] for k in sorted(multi_food_counter.keys())]
bar_colors = ["#4C72B0"] + ["#DD8452"] * (len(labels) - 1)

bars = ax1.bar(labels, values, color=bar_colors, edgecolor="white", linewidth=1.2)
ax1.set_xlabel("# Food Classes in Image", fontsize=12)
ax1.set_ylabel("# Images", fontsize=12)
ax1.set_title("Food Classes per Image (Training Set)", fontweight="bold", fontsize=14)
for b, v in zip(bars, values):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
             f"{v}\n({v/total_imgs*100:.1f}%)", ha="center", fontsize=11, fontweight="bold")

summary = f"""KEY INSIGHT

  {single_pct:.0f}% of images contain
  exactly 1 food class.

  Only {multi_pct:.1f}% have ≥ 2 foods.

  → Task: segment ONE food
  from background per image.
  Not multi-food separation.

  → Main challenge:
  distinguish 102 categories."""

ax2.text(0.05, 0.95, summary, transform=ax2.transAxes,
         fontsize=12, fontfamily="monospace", verticalalignment="top")
ax2.axis("off")

plt.tight_layout()
plt.savefig("eda_multi_food_analysis.png", dpi=150)
plt.show()

### 6.2 Food Coverage (Foreground Ratio)


In [ ]:
# ====== Food vs background pixel ratio ======
sample_n = min(2000, len(train_pairs))
sample_pairs = random.sample(train_pairs, sample_n)
food_ratios = []

for _, mask_path in tqdm(sample_pairs, desc="Computing coverage"):
    mask = read_mask(mask_path)
    food_px = (mask > 0).sum()
    food_ratios.append(food_px / mask.size * 100)

food_ratios = np.array(food_ratios)

print(f"Food coverage: mean={food_ratios.mean():.0f}%  median={np.median(food_ratios):.0f}%  "
      f"min={food_ratios.min():.0f}%  max={food_ratios.max():.0f}%")
print(f"→ ~{100 - food_ratios.mean():.0f}% background per image (normal for segmentation)")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(food_ratios, bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvline(food_ratios.mean(), color="red", linestyle="--", linewidth=2,
                label=f"Mean: {food_ratios.mean():.1f}%")
axes[0].axvline(np.median(food_ratios), color="orange", linestyle="--", linewidth=2,
                label=f"Median: {np.median(food_ratios):.1f}%")
axes[0].set_xlabel("Food Coverage (% of image pixels)", fontsize=12)
axes[0].set_ylabel("# Images", fontsize=12)
axes[0].set_title("Food Coverage Distribution", fontweight="bold", fontsize=14)
axes[0].legend(fontsize=10)

bp = axes[1].boxplot([food_ratios], vert=True, patch_artist=True, widths=0.4,
                     labels=["Food Coverage %"])
bp["boxes"][0].set_facecolor("#DD8452")
axes[1].set_ylabel("% of Image Pixels", fontsize=12)
axes[1].set_title("Food Coverage Box Plot", fontweight="bold", fontsize=14)

plt.tight_layout()
plt.savefig("eda_food_coverage.png", dpi=150)
plt.show()

---
## 7. Image Size Analysis


In [ ]:
# ====== Scan all image sizes ======
print("Scanning image dimensions ...")
widths, heights, aspects, areas = [], [], [], []
for img_path, _ in tqdm(train_pairs + test_pairs, desc="Scanning sizes"):
    with Image.open(img_path) as im:
        w, h = im.size
        widths.append(w); heights.append(h)
        aspects.append(w / h); areas.append(w * h)
widths = np.array(widths); heights = np.array(heights)
aspects = np.array(aspects); areas = np.array(areas)
print(f"\\nWidth:  min={widths.min():.0f}  max={widths.max():.0f}  mean={widths.mean():.0f}")
print(f"Height: min={heights.min():.0f}  max={heights.max():.0f}  mean={heights.mean():.0f}")
print(f"Aspect: min={aspects.min():.3f}  max={aspects.max():.3f}  mean={aspects.mean():.3f}")


In [ ]:
# ====== Image Size Histogram ======
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Width histogram ---
axes[0, 0].hist(widths, bins=60, color="#4C72B0", edgecolor='white', alpha=0.85)
axes[0, 0].axvline(widths.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {widths.mean():.0f}px")
axes[0, 0].axvline(np.median(widths), color='orange', linestyle='--', linewidth=2, label=f"Median: {np.median(widths):.0f}px")
axes[0, 0].set_xlabel("Width (px)", fontsize=12)
axes[0, 0].set_ylabel("# Images", fontsize=12)
axes[0, 0].set_title("Image Width Distribution", fontweight='bold', fontsize=14)
axes[0, 0].legend(fontsize=10)

# --- Height histogram ---
axes[0, 1].hist(heights, bins=60, color="#DD8452", edgecolor='white', alpha=0.85)
axes[0, 1].axvline(heights.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {heights.mean():.0f}px")
axes[0, 1].axvline(np.median(heights), color='orange', linestyle='--', linewidth=2, label=f"Median: {np.median(heights):.0f}px")
axes[0, 1].set_xlabel("Height (px)", fontsize=12)
axes[0, 1].set_ylabel("# Images", fontsize=12)
axes[0, 1].set_title("Image Height Distribution", fontweight='bold', fontsize=14)
axes[0, 1].legend(fontsize=10)

# --- Aspect ratio histogram ---
axes[1, 0].hist(aspects, bins=60, color="#55A868", edgecolor='white', alpha=0.85)
axes[1, 0].axvline(aspects.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {aspects.mean():.3f}")
axes[1, 0].set_xlabel("Aspect Ratio (W/H)", fontsize=12)
axes[1, 0].set_ylabel("# Images", fontsize=12)
axes[1, 0].set_title("Aspect Ratio Distribution", fontweight='bold', fontsize=14)
axes[1, 0].legend(fontsize=10)

# --- Width vs Height scatter ---
axes[1, 1].scatter(widths[::10], heights[::10], c='#6A51A3', alpha=0.35, s=12, edgecolors='none')
axes[1, 1].set_xlabel("Width (px)", fontsize=12)
axes[1, 1].set_ylabel("Height (px)", fontsize=12)
axes[1, 1].set_title("Width vs Height (sampled)", fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig("eda_image_size_histogram.png", dpi=150)
plt.show()


---
## 8. Random Sample Images


In [ ]:
# ====== Display random samples from training set ======
N_SAMPLES = 16
random.shuffle(train_pairs)
samples = train_pairs[:N_SAMPLES]

cols = 4
rows = (N_SAMPLES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4.2))
axes = axes.flatten()

for idx, (img_path, _) in enumerate(samples):
    img = Image.open(img_path)
    fname = os.path.basename(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(f"{fname}\n{img.size[0]}×{img.size[1]}", fontsize=9)
    axes[idx].axis('off')

# Hide unused subplots
for idx in range(N_SAMPLES, len(axes)):
    axes[idx].axis('off')

plt.suptitle("Random Training Samples", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("eda_sample_images.png", dpi=150)
plt.show()


---
## 9. Segmentation Mask Visualization


In [ ]:
# ====== Generate a colour map for background + all food classes ======
def make_class_colormap(num_classes):
    cmap = plt.colormaps["gist_ncar"]
    colours = (np.array([cmap(i / num_classes)[:3] for i in range(num_classes)]) * 255).astype(np.uint8)
    colours[0] = [0, 0, 0]   # background = black
    return colours

CLASS_COLORS = make_class_colormap(NUM_CLASSES + 1)   # +1 for background (class 0)

def mask_to_rgb(mask, colors):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cid in range(len(colors)):
        rgb[mask == cid] = colors[cid]
    return rgb

print(f"Colormap: {len(CLASS_COLORS)} colours (class 0 = background + {NUM_CLASSES} food classes)")


In [ ]:
# ====== Display: original | mask_colour | overlay ======
N_SHOW = 8
fig, axes = plt.subplots(N_SHOW, 3, figsize=(14, N_SHOW * 3.8))

col_titles = ["Original", "Segmentation Mask", "Overlay"]
for c in range(3):
    axes[0, c].set_title(col_titles[c], fontweight='bold', fontsize=13)

for row, (img_path, mask_path) in enumerate(samples[:N_SHOW]):
    # Read
    img  = np.array(Image.open(img_path).convert('RGB'))
    mask = read_mask(mask_path)
    mask_rgb = mask_to_rgb(mask, CLASS_COLORS)

    # Resize mask to match image (in case they differ)
    if mask.shape[:2] != img.shape[:2]:
        pil_mask = Image.fromarray(mask_rgb)
        mask_rgb = np.array(pil_mask.resize((img.shape[1], img.shape[0]), Image.NEAREST))
        mask = np.array(Image.fromarray(mask).resize((img.shape[1], img.shape[0]), Image.NEAREST))

    overlay = (img * 0.5 + mask_rgb * 0.5).astype(np.uint8)

    # Unique classes in this mask
    present = sorted(np.unique(mask))
    present_names = [ID2NAME.get(int(c), '?') for c in present if c != 0]
    label = ', '.join(present_names[:3])
    if len(present_names) > 3:
        label += f" (+{len(present_names)-3} more)"

    axes[row, 0].imshow(img)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_rgb)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(overlay)
    axes[row, 2].axis('off')

    # Put class label on the right
    axes[row, 1].set_ylabel(f"{os.path.basename(img_path)}\n{label}",
                            fontsize=8, rotation=0, ha='right', va='center',
                            labelpad=10)

plt.suptitle("Segmentation Mask Visualisation", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("eda_mask_visualisation.png", dpi=150)
plt.show()


---
## 10. Per-Class Image Frequency

How many training images does each class appear in? Sorted frequency chart to identify under-represented classes.


In [ ]:
# ====== Per-class image frequency chart ======
food_ids_list = sorted(ID2NAME.keys())
food_img_counts_freq = [class_image_counter.get(cid, 0) for cid in food_ids_list]

fig, ax = plt.subplots(figsize=(20, 6))

sorted_idx_food = np.argsort(food_img_counts_freq)[::-1]
sorted_fids  = [food_ids_list[i] for i in sorted_idx_food]
sorted_fnames = [ID2NAME[cid] for cid in sorted_fids]
sorted_fcnts  = [food_img_counts_freq[i] for i in sorted_idx_food]

colors_img = ["#E74C3C" if c < 60 else "#F39C12" if c < 90 else "#55A868"
              for c in sorted_fcnts]

ax.bar(range(len(sorted_fcnts)), sorted_fcnts, color=colors_img,
       width=0.7, edgecolor="white", linewidth=0.2)
ax.set_xlabel("Food Class (sorted by image count)", fontsize=12)
ax.set_ylabel("# Training Images containing this class", fontsize=12)
ax.set_title("Per-Class Image Frequency", fontweight="bold", fontsize=15)
ax.axhline(y=num_train / NUM_CLASSES, color="gray", linestyle=":", linewidth=1.5,
           label=f"Uniform: {num_train/NUM_CLASSES:.0f} images/class")

legend_elements = [
    mpatches.Patch(color="#55A868", label="\u2265 90 images"),
    mpatches.Patch(color="#F39C12", label="60–89 images"),
    mpatches.Patch(color="#E74C3C", label="< 60 images (tail)"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=10)

plt.tight_layout()
plt.savefig("eda_class_image_frequency.png", dpi=150)
plt.show()

# Print top-10 and bottom-10
print("Top 10 most frequent classes:")
for i in range(min(10, len(sorted_fids))):
    print(f"  {sorted_fids[i]:>3d} {sorted_fnames[i]:<35s} — {sorted_fcnts[i]:>5d} images  ({sorted_fcnts[i]/num_train*100:.1f}%)")
print()
print("Bottom 10 least frequent classes:")
for i in range(1, min(11, len(sorted_fids))):
    j = -i
    print(f"  {sorted_fids[j]:>3d} {sorted_fnames[j]:<35s} — {sorted_fcnts[j]:>5d} images  ({sorted_fcnts[j]/num_train*100:.1f}%)")


---
## 11. Summary & Recommendations for Modelling


In [ ]:
# ====== Comprehensive EDA Summary ======
print("=" * 55)
print("  UECFoodPix — EDA SUMMARY")
print("=" * 55)
print(f"  Images:    {total:,} ({num_train:,} train / {num_test:,} test) — 90/10 split")
print(f"  Classes:   {NUM_CLASSES} food + background = {NUM_CLASSES + 1} output channels")
print(f"  Per class: min={min(food_img_counts)}, max={max(food_img_counts)}, mean={np.mean(food_img_counts):.0f}, median={np.median(food_img_counts):.0f}")
print(f"  Task type: {single_pct:.0f}% single-food images → single-food segmentation")
print(f"  Coverage:  mean {food_ratios.mean():.0f}% food, {100-food_ratios.mean():.0f}% background per image")
print(f"  Size:      {widths.min():.0f}×{heights.min():.0f} ~ {widths.max():.0f}×{heights.max():.0f} px, median {np.median(widths):.0f}×{np.median(heights):.0f}")
print(f"  Balance:   {ratio:.0f}:1 max/min ({tail_60}/102 classes < 60 images) — mild imbalance")
print(f"  Mask:      Bounding box + GrabCut (auto-generated, not manual)")
print("-" * 55)
print("  RECOMMENDATIONS:")
print(f"    Input: 512×512  |  Norm: ImageNet  |  Arch: DeepLabV3+/SegFormer")
print(f"    Loss: CrossEntropy  |  Metric: mIoU  |  Aug: HFlip, Rot±15°, Jitter")
print(f"    Output: {NUM_CLASSES+1} channels (0=bg, 1–{NUM_CLASSES}=food)")
print("=" * 55)

---
## 12. Output Figures

| # | Figure | Description |
|---|---|---|
| 1 | `eda_dataset_overview.png` | Train/test split pie + summary |
| 2 | `eda_category_groups.png` | 13 food category groups & image counts |
| 3 | `eda_class_distribution.png` | Per-class image count (top-30 + all 102) |
| 4 | `eda_class_imbalance.png` | Image-level log-scale distribution + tail class pie |
| 5 | `eda_multi_food_analysis.png` | Food classes per image (88% single-food) |
| 6 | `eda_food_coverage.png` | Food vs background pixel ratio distribution |
| 7 | `eda_image_size_histogram.png` | Width / height / aspect ratio distributions |
| 8 | `eda_sample_images.png` | 16 random training samples |
| 9 | `eda_mask_visualisation.png` | 8\u00d7 (original, mask, overlay) with class labels |
| 10 | `eda_class_image_frequency.png` | Per-class image frequency bar chart |

> All figures are saved to the current working directory as 150 DPI PNGs.
>
> **EDA completed.** All findings feed directly into modelling decisions.
